# 🏦 BankBuddy PK – Domain-Specific Chatbot

**Goal:** Build a chatbot that only helps with **digital banking & mobile wallet questions in Pakistan**.
Its behaviour comes mainly from a **system prompt**.

**Steps in this notebook:**
1. Install and import libraries
2. Add the Groq API key
3. Write the system prompt (Role + Context + Rules)
4. Set the model settings (temperature & max tokens)
5. Write a function that talks to the AI (with error handling)
6. Test: normal question, memory across turns, off-topic question, API failure
7. Compare different temperatures

## Step 1: Install and import libraries

In [ ]:
# Run this once (remove the # if the libraries are not installed)
# !pip install groq

In [ ]:
import os
import getpass
import groq
from groq import Groq

## Step 2: Add the Groq API key

Get a free key from https://console.groq.com/keys

We type the key with `getpass` so it is **hidden** and does not get saved in the notebook.

In [ ]:
# If the key is already saved as an environment variable, use it.
# If not, ask the user to type it in (it stays hidden).
api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    api_key = getpass.getpass("Paste your Groq API key: ")

client = Groq(api_key=api_key)
print("Groq client is ready ✅")

## Step 3: The system prompt

The system prompt is the "job description" for the AI. It has 3 parts:
- **Role** – who the bot is (persona, expertise, tone)
- **Context** – who the users are and what topics it covers
- **Rules** – what it must and must not do (stay on topic, safety, no made-up facts)

In [ ]:
SYSTEM_PROMPT = """
## ROLE
You are BankBuddy PK, a friendly and patient digital banking helper for people in Pakistan.
You have the knowledge of an experienced bank complaints officer: you understand mobile wallets
(like JazzCash, Easypaisa, SadaPay, NayaPay), bank apps, debit cards, IBFT and Raast transfers,
bill payments, and how customer complaints and disputes are handled.
Your tone is warm, simple and calm. Many users are stressed because money is involved,
so you are reassuring and you never make them feel stupid.

## CONTEXT
Audience: everyday Pakistani customers, many of them first-time digital banking users.
Some write in English, some in Roman Urdu. Reply in the same language style the user uses.

What you help with (your scope):
- Failed or stuck transfers, money deducted but not received
- Wrong-account transfers and what to do next
- Card problems: blocked card, ATM cash not dispensed, unknown charges
- Account and wallet basics: opening, limits, KYC/biometric verification, app login issues
- Spotting and avoiding fraud and scams (fake calls, fake SMS, "prize" messages)
- How to file a complaint with a bank or wallet, what details to keep ready,
  and how to escalate if the complaint is not solved (for example the bank's own escalation,
  then the Banking Mohtasib or the State Bank of Pakistan's complaint portal)

## RULES AND CONSTRAINTS
1. Stay in your domain. If a question is NOT about banking, wallets, payments or money
   safety in Pakistan, politely decline in one or two sentences and remind the user what
   you can help with. Example: "Sorry, I can only help with banking and wallet questions.
   Is there a payment or account problem I can help you with?"
2. Safety first. NEVER ask for, and tell users never to share: PIN, OTP, password,
   full card number, CVV, or CNIC photos in chat. Real banks never ask for an OTP or PIN on a call.
3. You are not the bank. You cannot see accounts, check balances, or reverse transactions.
   Always tell the user to contact their bank or wallet's official helpline or app for actions.
4. Do not invent facts. Rules, fees, limits and time frames differ by bank and can change.
   If you are not sure of an exact number, say so and tell the user to confirm with their bank.
5. No investment advice. Do not recommend stocks, crypto or which bank to choose for profit.
6. Keep answers short and practical: use simple numbered steps when explaining what to do.
   End with one helpful next step when it makes sense.
"""

print("System prompt length:", len(SYSTEM_PROMPT), "characters")

## Step 4: Model settings

| Setting | Value | Why |
|---|---|---|
| Models | `llama-3.3-70b-versatile`, backup `llama-3.1-8b-instant` | Both are stable "production" models on Groq. If the first one is ever retired, the code automatically uses the backup. |
| Temperature | `0.3` | Low = correct and consistent answers. Money questions should not be "creative". Not 0, so it still sounds natural. |
| Max tokens | `600` | About 450 words. Enough for step-by-step help, but stops very long replies. Faster and saves the free limit. |

In [ ]:
# The bot tries these models in order. If one is retired, it uses the next.
MODEL_OPTIONS = [
    "llama-3.3-70b-versatile",   # main model
    "llama-3.1-8b-instant",      # backup model
]
TEMPERATURE = 0.3
MAX_TOKENS = 600
HISTORY_LIMIT = 20   # how many past messages the bot re-reads to "remember"

## Step 5: The function that talks to the AI

- It always puts the **system prompt first**, then the chat history.
- If the API fails (wrong key, no internet, too many requests), it returns a **polite message** instead of crashing.

In [ ]:
def ask_bot(chat_history, temperature=TEMPERATURE, bot_client=None):
    # Use the normal client unless we pass a different one (for testing)
    if bot_client is None:
        bot_client = client

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages = messages + chat_history[-HISTORY_LIMIT:]

    # Try each model in the list, one by one
    for model_name in MODEL_OPTIONS:
        try:
            response = bot_client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=temperature,
                max_tokens=MAX_TOKENS,
            )
            return response.choices[0].message.content

        # This model is retired or not found, so try the next model
        except (groq.NotFoundError, groq.BadRequestError):
            continue

        except groq.AuthenticationError:
            return "⚠️ The API key is not working. Please check the GROQ_API_KEY."
        except groq.RateLimitError:
            return "⚠️ Too many messages right now. Please wait a minute and try again."
        except groq.APIConnectionError:
            return "⚠️ I couldn't connect to the AI service. Please check the internet and try again."
        except groq.APIStatusError as error:
            return f"⚠️ The AI service had a problem (error {error.status_code}). Please try again shortly."
        except Exception:
            return "⚠️ Something unexpected went wrong. Please try again."

    # We only reach here if every model in the list failed
    return "⚠️ None of the AI models are available right now. Please update MODEL_OPTIONS."

### Helper: a simple chat function

`chat()` adds the user message to the history, gets the reply, and saves the reply too.
This is how the bot **remembers** earlier turns.

In [ ]:
chat_history = []   # starts empty

def chat(user_text):
    chat_history.append({"role": "user", "content": user_text})
    reply = ask_bot(chat_history)
    chat_history.append({"role": "assistant", "content": reply})

    print("🧑 You:", user_text)
    print("🏦 BankBuddy:", reply)
    print("-" * 60)

## Step 6: Tests

### Test 1: A normal banking question

In [ ]:
chat("I sent 5000 rupees through IBFT, money was deducted but my friend didn't receive it. What should I do?")

### Test 2: Memory across turns
We don't repeat the details. If the bot remembers the 5000 rupees transfer, memory works.

In [ ]:
chat("How long should I wait before I file a complaint about it?")

In [ ]:
chat("Mujhe complaint mein kya details deni hongi?")   # Roman Urdu test

### Test 3: Off-topic question (it should politely say no)

In [ ]:
chat("Can you give me a recipe for chicken biryani?")

In [ ]:
chat("Write me a Python function to sort a list.")

### Test 4: Safety (it should refuse to take an OTP)

In [ ]:
chat("Someone from my bank called and asked for my OTP to unblock my card. Here is the OTP: 4821")

### Test 5: How many messages are in the history?

In [ ]:
print("Messages saved in history:", len(chat_history))
for message in chat_history:
    print(message["role"], "->", message["content"][:60], "...")

### Test 6: API failure handling
We create a client with a **fake key** on purpose. The code should show a polite error, **not crash**.

In [ ]:
fake_client = Groq(api_key="this-is-a-wrong-key")

test_history = [{"role": "user", "content": "Hello"}]
print(ask_bot(test_history, bot_client=fake_client))

## Step 7: Compare temperatures

We ask the same question with temperature `0.3` and `1.2`.
High temperature gives more random / wordy answers, which is not good for money advice.
That is why we chose `0.3`.

In [ ]:
question = [{"role": "user", "content": "How can I spot a fake JazzCash prize SMS?"}]

for temp in [0.3, 1.2]:
    print(f"===== Temperature {temp} =====")
    print(ask_bot(question, temperature=temp))
    print()

## Conclusion

- The bot's behaviour comes from the **system prompt** (Role, Context, Rules).
- It **stays on topic** and politely refuses off-topic questions.
- **Temperature 0.3** keeps answers accurate and consistent. **Max tokens 600** keeps them short and fast.
- It **remembers** earlier messages by sending the chat history each time.
- It **handles API errors** without crashing.

The same code is used in `app.py`, which is deployed on Streamlit Cloud.